# LeetCode 1878 — Get Biggest Three Rhombus Sums in a Grid bằng Python

Link đề: https://leetcode.com/problems/get-biggest-three-rhombus-sums-in-a-grid/ · Độ khó: **Medium** · Chủ đề: Array, Math, Sorting, Heap (Priority Queue), Matrix, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def getBiggestThree(self, grid: List[List[int]]) -> List[int]:
        ...
```

Lộ trình:
1. **Vét cạn theo định nghĩa** — với mỗi hình thoi (đỉnh trên + bán kính), quét **cả lưới** và cộng các ô có khoảng cách Manhattan tới tâm đúng bằng bán kính (`O(m n K · m n)`, dùng để kiểm chứng).
2. **Đi dọc 4 cạnh** của từng hình thoi (`O(m n K²)`) — đủ nhanh với lưới `50 × 50`.
3. **Tổng tiền tố theo hai đường chéo** — mỗi hình thoi tính trong `O(1)` ⇒ `O(m n K)`.
4. Giữ **3 giá trị phân biệt lớn nhất** bằng `set` + sắp xếp, bằng 3 biến cập nhật tại chỗ, hoặc bằng min-heap kích thước 3.

## 1. Mục tiêu học

1. Mô tả hình học **hình thoi** trên lưới (hình vuông xoay 45°) bằng **đỉnh trên** `(r, c)` và **bán kính** `k`: bốn đỉnh `(r, c)`, `(r+k, c+k)`, `(r+2k, c)`, `(r+k, c−k)`; điều kiện nằm trong lưới.
2. Liệt kê các ô trên **biên** hình thoi theo 4 cạnh chéo, và tránh **đếm trùng 4 đỉnh**.
3. Xây **tổng tiền tố theo đường chéo** `↘` và `↙`, lấy tổng một đoạn chéo trong `O(1)`.
4. Duy trì **top-3 giá trị phân biệt** — chú ý chữ "distinct" và trường hợp có ít hơn 3 giá trị.
5. Kiểm chứng bằng định nghĩa "khoảng cách Manhattan tới tâm = `k`" — một cách mô tả hoàn toàn khác.

## 2. Đề bài & ví dụ

Cho ma trận số nguyên `grid` kích thước `m × n`. Một **tổng hình thoi** là tổng các phần tử nằm trên **biên** của một hình thoi đều trong lưới: hình thoi có dạng hình vuông xoay 45°, mỗi góc đặt **đúng tâm một ô** của lưới. Hình thoi có thể có diện tích 0 (chỉ là một ô).

Trả về **ba tổng hình thoi phân biệt lớn nhất** theo thứ tự giảm dần. Nếu có ít hơn ba giá trị phân biệt, trả về tất cả.

**Ví dụ 1:** `grid = [[3,4,5,1,3],[3,3,4,2,3],[20,30,200,40,10],[1,5,5,4,1],[4,3,2,2,5]]` → `[228,216,211]`.
- Xanh dương: `20 + 3 + 200 + 5 = 228`; Đỏ: `200 + 2 + 10 + 4 = 216`; Xanh lá: `5 + 200 + 4 + 2 = 211`.

**Ví dụ 2:** `grid = [[1,2,3],[4,5,6],[7,8,9]]` → `[20,9,8]` (`4 + 2 + 6 + 8 = 20`; ô đơn `9`; ô đơn `8`).

**Ví dụ 3:** `grid = [[7,7,7]]` → `[7]`.

**Ràng buộc:** `1 ≤ m, n ≤ 50`, `1 ≤ grid[i][j] ≤ 10⁵`.

In [1]:
import random
import time
import heapq
from typing import List

G1 = [[3, 4, 5, 1, 3], [3, 3, 4, 2, 3], [20, 30, 200, 40, 10], [1, 5, 5, 4, 1], [4, 3, 2, 2, 5]]
G2 = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
G3 = [[7, 7, 7]]
examples = [(G1, [228, 216, 211]), (G2, [20, 9, 8]), (G3, [7])]
for g, want in examples:
    print(f"{len(g)}x{len(g[0])} -> kỳ vọng {want}")

5x5 -> kỳ vọng [228, 216, 211]
3x3 -> kỳ vọng [20, 9, 8]
1x3 -> kỳ vọng [7]


## 3. Kiến thức nền

### 3.1 Hình thoi = "đường tròn" theo khoảng cách Manhattan

Gọi tâm hình thoi là `(cr, cc)` và bán kính `k`. Biên hình thoi chính là các ô `(i, j)` có `|i − cr| + |j − cc| = k`. Với `k = 0` chỉ có ô tâm. Khi biểu diễn bằng **đỉnh trên** `(r, c)` thì tâm là `(r + k, c)`.

In [2]:
def border_cells_by_manhattan(m, n, cr, cc, k):
    return [(i, j) for i in range(m) for j in range(n) if abs(i - cr) + abs(j - cc) == k]

cells = border_cells_by_manhattan(5, 5, 2, 2, 2)
print(len(cells), "ô trên biên hình thoi bán kính 2 (= 4k):", cells)

8 ô trên biên hình thoi bán kính 2 (= 4k): [(0, 2), (1, 1), (1, 3), (2, 0), (2, 4), (3, 1), (3, 3), (4, 2)]


### 3.2 Điều kiện hình thoi nằm trong lưới

Đỉnh trên `(r, c)`, bán kính `k`: cần `r + 2k ≤ m − 1` (đỉnh dưới), `c − k ≥ 0` (đỉnh trái), `c + k ≤ n − 1` (đỉnh phải). Bán kính lớn nhất `K` khoảng `min(m, n) / 2`.

In [3]:
def fits(m, n, r, c, k):
    return r + 2 * k <= m - 1 and c - k >= 0 and c + k <= n - 1

m, n = 5, 5
print([(k, fits(m, n, 0, 2, k)) for k in range(4)])
total = sum(1 for r in range(m) for c in range(n) for k in range(26) if fits(m, n, r, c, k))
print("số hình thoi trong lưới 5x5:", total)

[(0, True), (1, True), (2, True), (3, False)]
số hình thoi trong lưới 5x5: 35


### 3.3 Bốn cạnh chéo của hình thoi

Từ đỉnh trên `T = (r, c)`:
- cạnh `T → phải`: `(r + t, c + t)`, `t = 0..k` (hướng `↘`),
- cạnh `phải → dưới`: `(r + k + t, c + k − t)` (hướng `↙`),
- cạnh `trái → dưới`: `(r + k + t, c − k + t)` (hướng `↘`),
- cạnh `T → trái`: `(r + t, c − t)` (hướng `↙`).

Mỗi cạnh có `k + 1` ô; bốn đỉnh bị đếm hai lần ⇒ tổng ô biên `4(k + 1) − 4 = 4k` (với `k ≥ 1`).

In [4]:
def border_cells_walk(r, c, k):
    if k == 0:
        return [(r, c)]
    cells = []
    for t in range(k):
        cells.append((r + t, c + t))              # T -> phải (không lấy đỉnh phải)
        cells.append((r + k + t, c + k - t))      # phải -> dưới
        cells.append((r + 2 * k - t, c - t))      # dưới -> trái
        cells.append((r + k - t, c - k + t))      # trái -> T
    return cells

cw = border_cells_walk(0, 2, 2)
print(len(cw), sorted(cw) == sorted(border_cells_by_manhattan(5, 5, 2, 2, 2)))

8 True


Mẹo tránh đếm trùng: đi vòng quanh, mỗi cạnh lấy `k` ô (bỏ ô cuối — chính là ô đầu của cạnh kế tiếp).

### 3.4 Tổng tiền tố theo đường chéo

- Chéo `↘`: `D[i + 1][j + 1] = D[i][j] + g[i][j]`. Tổng đoạn từ `(i1, j1)` xuống `(i2, j2)` (cùng chéo, `i2 ≥ i1`) là `D[i2 + 1][j2 + 1] − D[i1][j1]`.
- Chéo `↙`: `A[i + 1][j] = A[i][j + 1] + g[i][j]` (mảng kích thước `(m + 1) × (n + 1)`). Tổng đoạn từ `(i1, j1)` xuống trái tới `(i2, j2)` là `A[i2 + 1][j2] − A[i1][j1 + 1]`.

In [5]:
def diag_prefix(g):
    m, n = len(g), len(g[0])
    D = [[0] * (n + 1) for _ in range(m + 1)]
    A = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m):
        for j in range(n):
            D[i + 1][j + 1] = D[i][j] + g[i][j]
            A[i + 1][j] = A[i][j + 1] + g[i][j]
    return D, A

D, A = diag_prefix(G2)
print("↘ (0,0)->(2,2):", D[3][3] - D[0][0], "= 1+5+9")
print("↙ (0,2)->(2,0):", A[3][0] - A[0][3], "= 3+5+7")

↘ (0,0)->(2,2): 15 = 1+5+9
↙ (0,2)->(2,0): 15 = 3+5+7


### 3.5 Top-3 giá trị phân biệt

`sorted(set(values), reverse=True)[:3]` — đơn giản nhất. Hoặc duy trì ba biến `a > b > c` và chèn giá trị mới nếu nó khác cả ba.

In [6]:
vals = [5, 9, 9, 7, 5, 3, 9]
print(sorted(set(vals), reverse=True)[:3])

[9, 7, 5]


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 2 `[[1,2,3],[4,5,6],[7,8,9]]` (3×3):
- 9 hình thoi bán kính 0: các ô `1..9`.
- Hình thoi bán kính 1 chỉ có thể có đỉnh trên `(0, 1)` (cần `r + 2 ≤ 2`, `1 ≤ c ≤ 1`): biên `2, 6, 8, 4` ⇒ `20`.

Tập các tổng `{1,…,9, 20}` ⇒ ba giá trị lớn nhất phân biệt: `20, 9, 8`. ✓

Ví dụ 3 `[[7,7,7]]`: chỉ có hình thoi bán kính 0 (lưới 1 hàng) ⇒ mọi tổng là 7 ⇒ `[7]` (chỉ một giá trị phân biệt).

In [7]:
g = G2
for r in range(3):
    for c in range(3):
        for k in range(2):
            if fits(3, 3, r, c, k):
                cells = border_cells_walk(r, c, k)
                print(f"đỉnh trên ({r},{c}), k={k}: ô {cells} -> tổng {sum(g[i][j] for i, j in cells)}")

đỉnh trên (0,0), k=0: ô [(0, 0)] -> tổng 1
đỉnh trên (0,1), k=0: ô [(0, 1)] -> tổng 2
đỉnh trên (0,1), k=1: ô [(0, 1), (1, 2), (2, 1), (1, 0)] -> tổng 20
đỉnh trên (0,2), k=0: ô [(0, 2)] -> tổng 3
đỉnh trên (1,0), k=0: ô [(1, 0)] -> tổng 4
đỉnh trên (1,1), k=0: ô [(1, 1)] -> tổng 5
đỉnh trên (1,2), k=0: ô [(1, 2)] -> tổng 6
đỉnh trên (2,0), k=0: ô [(2, 0)] -> tổng 7
đỉnh trên (2,1), k=0: ô [(2, 1)] -> tổng 8
đỉnh trên (2,2), k=0: ô [(2, 2)] -> tổng 9


## 5. Cách 1 — Brute Force / baseline theo định nghĩa Manhattan

**Ý tưởng:** với mỗi hình thoi hợp lệ (đỉnh trên `(r, c)`, bán kính `k`), duyệt **mọi ô** của lưới và cộng ô có `|i − (r + k)| + |j − c| = k`. Không cần nghĩ về cạnh hay đỉnh — rất khó sai, dùng để kiểm chứng.

In [8]:
def biggest_three_manhattan(grid):
    m, n = len(grid), len(grid[0])
    sums = set()
    for r in range(m):
        for c in range(n):
            k = 0
            while fits(m, n, r, c, k):
                cr = r + k
                s = 0
                for i in range(m):
                    for j in range(n):
                        if abs(i - cr) + abs(j - c) == k:
                            s += grid[i][j]
                sums.add(s)
                k += 1
    return sorted(sums, reverse=True)[:3]

for g, want in examples:
    print(biggest_three_manhattan(g), "kỳ vọng", want)

[228, 216, 211] kỳ vọng [228, 216, 211]
[20, 9, 8] kỳ vọng [20, 9, 8]
[7] kỳ vọng [7]


**Độ phức tạp:** số hình thoi `O(m n K)` (`K ≈ min(m, n)/2`), mỗi hình thoi quét `m n` ô ⇒ `O(m² n² K)` — với `50 × 50` khoảng `50²·50²·25 ≈ 1.5·10⁸`: quá chậm trong Python, chỉ dùng cho lưới nhỏ.

**Nút thắt:** biên hình thoi chỉ có `4k` ô, nhưng ta quét cả `m n` ô.

In [9]:
random.seed(0)
for s in [8, 12, 16]:
    g = [[random.randint(1, 100) for _ in range(s)] for _ in range(s)]
    t0 = time.perf_counter(); biggest_three_manhattan(g)
    print(f"{s}x{s}: {time.perf_counter() - t0:.3f}s")

8x8: 0.000s
12x12: 0.002s
16x16: 0.009s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (chỉ đi biên):** liệt kê trực tiếp `4k` ô trên biên (mục 3.3) ⇒ mỗi hình thoi `O(k)` ⇒ tổng `O(m n K²)` — với lưới `50 × 50`: khoảng `2500 · Σ_{k ≤ 25} 4k ≈ 3·10⁶` phép, chạy tốt.

**Insight 2 (tiền tố chéo):** mỗi cạnh là một **đoạn liên tiếp trên một đường chéo** ⇒ tổng `O(1)` với tiền tố chéo. Tổng biên = tổng 4 cạnh (mỗi cạnh `k + 1` ô, gồm cả hai đầu) − tổng 4 đỉnh.

**Insight 3 (top-3 phân biệt):** không cần lưu mọi tổng; chỉ giữ 3 giá trị phân biệt lớn nhất.

**Quy tắc:** với `k = 0`, tổng là `grid[r][c]`; với `k ≥ 1`, dùng công thức 4 cạnh − 4 đỉnh. Dừng tăng `k` khi hình thoi ra khỏi lưới.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Đi dọc biên từng hình thoi

In [10]:
def biggest_three_walk(grid):
    m, n = len(grid), len(grid[0])
    sums = set()
    for r in range(m):
        for c in range(n):
            sums.add(grid[r][c])
            k = 1
            while r + 2 * k < m and c - k >= 0 and c + k < n:
                s = 0
                for t in range(k):
                    s += grid[r + t][c + t]
                    s += grid[r + k + t][c + k - t]
                    s += grid[r + 2 * k - t][c - t]
                    s += grid[r + k - t][c - k + t]
                sums.add(s)
                k += 1
    return sorted(sums, reverse=True)[:3]

for g, want in examples:
    print(biggest_three_walk(g), "kỳ vọng", want)

[228, 216, 211] kỳ vọng [228, 216, 211]
[20, 9, 8] kỳ vọng [20, 9, 8]
[7] kỳ vọng [7]


Code nộp LeetCode của Cách 2:

```python
class Solution:
    def getBiggestThree(self, grid: List[List[int]]) -> List[int]:
        m, n = len(grid), len(grid[0])
        sums = set()
        for r in range(m):
            for c in range(n):
                sums.add(grid[r][c])
                k = 1
                while r + 2 * k < m and c - k >= 0 and c + k < n:
                    s = 0
                    for t in range(k):
                        s += grid[r + t][c + t] + grid[r + k + t][c + k - t]
                        s += grid[r + 2 * k - t][c - t] + grid[r + k - t][c - k + t]
                    sums.add(s)
                    k += 1
        return sorted(sums, reverse=True)[:3]
```

**Độ phức tạp:** Time `O(m n K²)`, Space `O(m n K)` cho tập tổng (có thể giảm còn `O(1)` bằng top-3).

### 7.2 Cách 3 — Tiền tố theo hai đường chéo, mỗi hình thoi `O(1)`

Từng đoạn với đỉnh trên `T = (r, c)`, phải `R = (r+k, c+k)`, dưới `B = (r+2k, c)`, trái `L = (r+k, c−k)`:
1. Cạnh `T → R` (`↘`): `D[r+k+1][c+k+1] − D[r][c]`.
2. Cạnh `L → B` (`↘`): `D[r+2k+1][c+1] − D[r+k][c−k]`.
3. Cạnh `T → L` (`↙`): `A[r+k+1][c−k] − A[r][c+1]`.
4. Cạnh `R → B` (`↙`): `A[r+2k+1][c] − A[r+k][c+k+1]`.
5. Trừ 4 đỉnh: `g[T] + g[R] + g[B] + g[L]`.

In [11]:
def biggest_three_prefix(grid):
    m, n = len(grid), len(grid[0])
    D, A = diag_prefix(grid)
    sums = set()
    for r in range(m):
        for c in range(n):
            sums.add(grid[r][c])
            k = 1
            while r + 2 * k < m and c - k >= 0 and c + k < n:
                tr = D[r + k + 1][c + k + 1] - D[r][c]
                lb = D[r + 2 * k + 1][c + 1] - D[r + k][c - k]
                tl = A[r + k + 1][c - k] - A[r][c + 1]
                rb = A[r + 2 * k + 1][c] - A[r + k][c + k + 1]
                corners = grid[r][c] + grid[r + k][c + k] + grid[r + 2 * k][c] + grid[r + k][c - k]
                sums.add(tr + lb + tl + rb - corners)
                k += 1
    return sorted(sums, reverse=True)[:3]

for g, want in examples:
    print(biggest_three_prefix(g), "kỳ vọng", want)

[228, 216, 211] kỳ vọng [228, 216, 211]
[20, 9, 8] kỳ vọng [20, 9, 8]
[7] kỳ vọng [7]


**Độ phức tạp:** tiền tố `O(m n)`, mỗi hình thoi `O(1)` ⇒ `O(m n K)`, Space `O(m n)`.

### 7.3 Cách 4 — Top-3 phân biệt bằng ba biến (không lưu tập)

Hàm `push(v)` chèn `v` vào bộ ba `(a, b, c)` giảm dần nếu `v` chưa có; giá trị "trống" là `0` (mọi tổng `≥ 1`).

In [12]:
def biggest_three_vars(grid):
    m, n = len(grid), len(grid[0])
    D, A = diag_prefix(grid)
    top = [0, 0, 0]                          # a >= b >= c, 0 = trống

    def push(v):
        if v in top:
            return
        if v > top[0]:
            top[0], top[1], top[2] = v, top[0], top[1]
        elif v > top[1]:
            top[1], top[2] = v, top[1]
        elif v > top[2]:
            top[2] = v

    for r in range(m):
        for c in range(n):
            push(grid[r][c])
            k = 1
            while r + 2 * k < m and c - k >= 0 and c + k < n:
                s = (D[r + k + 1][c + k + 1] - D[r][c] + D[r + 2 * k + 1][c + 1] - D[r + k][c - k]
                     + A[r + k + 1][c - k] - A[r][c + 1] + A[r + 2 * k + 1][c] - A[r + k][c + k + 1]
                     - grid[r][c] - grid[r + k][c + k] - grid[r + 2 * k][c] - grid[r + k][c - k])
                push(s)
                k += 1
    return [v for v in top if v > 0]

for g, want in examples:
    print(biggest_three_vars(g), "kỳ vọng", want)

[228, 216, 211] kỳ vọng [228, 216, 211]
[20, 9, 8] kỳ vọng [20, 9, 8]
[7] kỳ vọng [7]


Lưu ý kiểm tra `v in top` **trước** khi chèn — đây là cách bảo đảm "phân biệt". Dùng `0` làm "trống" an toàn vì mọi phần tử `≥ 1`.

**Độ phức tạp:** Time `O(m n K)`, Space `O(m n)` cho tiền tố, `O(1)` cho top-3.

### 7.4 Cách 5 — Min-heap kích thước 3 + tập đang giữ

Biến thể tổng quát cho "top-`t` phân biệt": heap chứa tối đa `t` giá trị, cùng một `set` các giá trị trong heap để kiểm tra trùng.

In [13]:
def top_t_distinct(values, t=3):
    h, inside = [], set()
    for v in values:
        if v in inside:
            continue
        if len(h) < t:
            heapq.heappush(h, v); inside.add(v)
        elif v > h[0]:
            out = heapq.heapreplace(h, v); inside.discard(out); inside.add(v)
    return sorted(h, reverse=True)

def all_rhombus_sums(grid):
    m, n = len(grid), len(grid[0]); D, A = diag_prefix(grid)
    for r in range(m):
        for c in range(n):
            yield grid[r][c]
            k = 1
            while r + 2 * k < m and c - k >= 0 and c + k < n:
                yield (D[r + k + 1][c + k + 1] - D[r][c] + D[r + 2 * k + 1][c + 1] - D[r + k][c - k]
                       + A[r + k + 1][c - k] - A[r][c + 1] + A[r + 2 * k + 1][c] - A[r + k][c + k + 1]
                       - grid[r][c] - grid[r + k][c + k] - grid[r + 2 * k][c] - grid[r + k][c - k])
                k += 1

def biggest_three_heap(grid):
    return top_t_distinct(all_rhombus_sums(grid), 3)

for g, want in examples:
    print(biggest_three_heap(g), "kỳ vọng", want)

[228, 216, 211] kỳ vọng [228, 216, 211]
[20, 9, 8] kỳ vọng [20, 9, 8]
[7] kỳ vọng [7]


**Độ phức tạp:** `O(m n K log t)` với `t = 3` ⇒ như Cách 3.

## 8. TRACE từng bước

Trace Cách 3 cho ví dụ 1 — in các hình thoi bán kính `k ≥ 1` cùng 4 cạnh và 4 đỉnh, chỉ những hình thoi có tổng `≥ 200`:

In [14]:
def trace_prefix(grid, threshold=200):
    m, n = len(grid), len(grid[0]); D, A = diag_prefix(grid)
    for r in range(m):
        for c in range(n):
            k = 1
            while r + 2 * k < m and c - k >= 0 and c + k < n:
                tr = D[r + k + 1][c + k + 1] - D[r][c]
                lb = D[r + 2 * k + 1][c + 1] - D[r + k][c - k]
                tl = A[r + k + 1][c - k] - A[r][c + 1]
                rb = A[r + 2 * k + 1][c] - A[r + k][c + k + 1]
                corners = grid[r][c] + grid[r + k][c + k] + grid[r + 2 * k][c] + grid[r + k][c - k]
                s = tr + lb + tl + rb - corners
                if s >= threshold:
                    print(f"đỉnh trên ({r},{c}) k={k}: T→R={tr}, L→B={lb}, T→L={tl}, R→B={rb}, "
                          f"đỉnh={corners} -> tổng {s}")
                k += 1

trace_prefix(G1)

đỉnh trên (0,2) k=1: T→R=7, L→B=203, T→L=8, R→B=202, đỉnh=210 -> tổng 210
đỉnh trên (1,1) k=1: T→R=203, L→B=25, T→L=23, R→B=205, đỉnh=228 -> tổng 228
đỉnh trên (1,3) k=1: T→R=12, L→B=204, T→L=202, R→B=14, đỉnh=216 -> tổng 216
đỉnh trên (2,2) k=1: T→R=204, L→B=7, T→L=205, R→B=6, đỉnh=211 -> tổng 211


Bảng đối chiếu với giải thích của đề (ví dụ 1):

| màu | đỉnh trên | k | các ô biên | tổng |
|---|---|---|---|---|
| xanh dương | (1, 1) | 1 | 3, 200, 5, 20 | 228 |
| đỏ | (1, 3) | 1 | 2, 10, 4, 200 | 216 |
| xanh lá | (2, 2) | 1 | 200, 4, 2, 5 | 211 |

(Hình thoi đỉnh trên `(0, 2)`, `k = 1` có tổng `210` — chỉ kém một chút, nên không lọt top-3.)

In [15]:
def trace_walk_cells(grid, r, c, k):
    cells = border_cells_walk(r, c, k)
    vals = [grid[i][j] for i, j in cells]
    print(f"đỉnh trên ({r},{c}) k={k}: ô {cells} giá trị {vals} -> tổng {sum(vals)}")

for r, c in [(1, 1), (1, 3), (2, 2)]:
    trace_walk_cells(G1, r, c, 1)
trace_walk_cells(G1, 0, 2, 2)

đỉnh trên (1,1) k=1: ô [(1, 1), (2, 2), (3, 1), (2, 0)] giá trị [3, 200, 5, 20] -> tổng 228
đỉnh trên (1,3) k=1: ô [(1, 3), (2, 4), (3, 3), (2, 2)] giá trị [2, 10, 4, 200] -> tổng 216
đỉnh trên (2,2) k=1: ô [(2, 2), (3, 3), (4, 2), (3, 1)] giá trị [200, 4, 2, 5] -> tổng 211
đỉnh trên (0,2) k=2: ô [(0, 2), (2, 4), (4, 2), (2, 0), (1, 3), (3, 3), (3, 1), (1, 1)] giá trị [5, 10, 2, 20, 2, 4, 5, 3] -> tổng 51


## 9. Code nộp LeetCode

In [16]:
class Solution:
    def getBiggestThree(self, grid: List[List[int]]) -> List[int]:
        m, n = len(grid), len(grid[0])
        D = [[0] * (n + 1) for _ in range(m + 1)]
        A = [[0] * (n + 1) for _ in range(m + 1)]
        for i in range(m):
            for j in range(n):
                D[i + 1][j + 1] = D[i][j] + grid[i][j]
                A[i + 1][j] = A[i][j + 1] + grid[i][j]
        sums = set()
        for r in range(m):
            for c in range(n):
                sums.add(grid[r][c])
                k = 1
                while r + 2 * k < m and c - k >= 0 and c + k < n:
                    s = (D[r + k + 1][c + k + 1] - D[r][c]
                         + D[r + 2 * k + 1][c + 1] - D[r + k][c - k]
                         + A[r + k + 1][c - k] - A[r][c + 1]
                         + A[r + 2 * k + 1][c] - A[r + k][c + k + 1]
                         - grid[r][c] - grid[r + k][c + k] - grid[r + 2 * k][c] - grid[r + k][c - k])
                    sums.add(s)
                    k += 1
        return sorted(sums, reverse=True)[:3]

sol = Solution()
for g, want in examples:
    got = sol.getBiggestThree(g)
    print(got, "OK" if got == want else "SAI")

[228, 216, 211] OK
[20, 9, 8] OK
[7] OK


## 10. Vì sao thuật toán đúng

**Liệt kê đủ mọi hình thoi.** Một hình thoi hợp lệ được xác định duy nhất bởi đỉnh trên `(r, c)` và bán kính `k ≥ 0`, với điều kiện cả bốn đỉnh trong lưới. Vòng lặp duyệt mọi `(r, c)` và tăng `k` cho tới khi điều kiện sai — vì khi `k` tăng, các đỉnh chỉ đi ra xa hơn, điều kiện **đơn điệu**, nên dừng ở `k` đầu tiên không hợp lệ là đúng.

**Biên = 4 cạnh trừ 4 đỉnh.** Với `k ≥ 1`, biên gồm 4 đoạn chéo mỗi đoạn `k + 1` ô, đôi một chỉ chung đúng một đỉnh (2 đoạn kề nhau chung một đỉnh, 2 đoạn đối diện không chung ô nào vì `k ≥ 1`). Theo bao hàm – loại trừ, tổng biên = tổng 4 đoạn − tổng 4 đỉnh. Với `k = 0` bốn "đỉnh" trùng nhau, nên phải xử lý riêng (`grid[r][c]`).

**Tiền tố chéo.** `D[i+1][j+1] = Σ_{t ≥ 0} g[i − t][j − t]` (dừng ở biên) ⇒ hiệu `D[i2+1][j2+1] − D[i1][j1]` đúng bằng tổng đoạn `(i1, j1) … (i2, j2)` trên chéo `↘`. Tương tự `A[i+1][j] = Σ g[i − t][j + t]` cho chéo `↙`.

**Top-3 phân biệt.** Tập `sums` chứa mọi tổng (không trùng); sắp giảm và lấy 3 phần tử đầu.

## 11. Phân tích Time / Space

Gọi `m, n ≤ 50`, `K = ⌊(min(m, n) − 1)/2⌋ + 1` số bán kính tối đa.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Manhattan vét cạn | `O(m² n² K)` | `O(mnK)` | lưới ≤ 16×16 |
| 2. Đi biên | `O(m n K²)` | `O(mnK)` | ~3·10⁶ với 50×50, đủ nhanh |
| 3. Tiền tố chéo | `O(m n K)` | `O(m n)` | **nộp bài** |
| 4. Tiền tố + 3 biến | `O(m n K)` | `O(m n)` | không lưu tập tổng |
| 5. Heap top-`t` | `O(m n K log t)` | `O(m n + t)` | top-`t` tổng quát |

In [17]:
random.seed(1)
g = [[random.randint(1, 10**5) for _ in range(50)] for _ in range(50)]
for name, fn in [("walk", biggest_three_walk), ("prefix", biggest_three_prefix), ("vars", biggest_three_vars),
                 ("heap", biggest_three_heap), ("submit", Solution().getBiggestThree)]:
    t0 = time.perf_counter(); r = fn(g)
    print(f"{name:6s}: {r}  {time.perf_counter() - t0:.3f}s")
print("số hình thoi trong 50x50:", sum(1 for _ in all_rhombus_sums(g)))

walk  : [5050163, 5008198, 4975009]  0.024s
prefix: [5050163, 5008198, 4975009]  0.012s
vars  : [5050163, 5008198, 4975009]  0.009s
heap  : [5050163, 5008198, 4975009]  0.008s
submit: [5050163, 5008198, 4975009]  0.010s
số hình thoi trong 50x50: 22100


## 12. Bộ test

So mọi cách với định nghĩa Manhattan trên lưới ngẫu nhiên nhỏ (đủ mọi hình dạng: 1 hàng, 1 cột, chữ nhật dẹt), giá trị nhỏ để có nhiều tổng trùng (kiểm tra "phân biệt").

In [18]:
fns = [biggest_three_walk, biggest_three_prefix, biggest_three_vars, biggest_three_heap,
       lambda g: Solution().getBiggestThree(g)]

for g, want in examples:
    assert biggest_three_manhattan(g) == want
    for fn in fns:
        assert fn(g) == want

edge = [([[5]], [5]), ([[1, 2]], [2, 1]), ([[1], [2], [3]], [3, 2, 1]), ([[4, 4], [4, 4]], [4]),
        ([[1, 1, 1], [1, 1, 1], [1, 1, 1]], [4, 1]), ([[100000] * 3] * 3, [400000, 100000])]
for g, want in edge:
    assert biggest_three_manhattan(g) == want, g
    for fn in fns:
        assert fn(g) == want, (fn, g)

random.seed(2024)
for _ in range(400):
    m, n = random.randint(1, 7), random.randint(1, 7)
    hi = random.choice([1, 2, 5, 100, 10**5])
    g = [[random.randint(1, hi) for _ in range(n)] for _ in range(m)]
    want = biggest_three_manhattan(g)
    for fn in fns:
        assert fn(g) == want, (fn, g)
print("ví dụ + edge + 400 lưới ngẫu nhiên khớp định nghĩa Manhattan")

ví dụ + edge + 400 lưới ngẫu nhiên khớp định nghĩa Manhattan

In [19]:
# kiểm tra riêng công thức tiền tố chéo cho TỪNG hình thoi (không chỉ top-3)
random.seed(9)
for _ in range(100):
    m, n = random.randint(1, 9), random.randint(1, 9)
    g = [[random.randint(1, 50) for _ in range(n)] for _ in range(m)]
    D, A = diag_prefix(g)
    for r in range(m):
        for c in range(n):
            k = 1
            while fits(m, n, r, c, k):
                walk = sum(g[i][j] for i, j in border_cells_walk(r, c, k))
                manh = sum(g[i][j] for i, j in border_cells_by_manhattan(m, n, r + k, c, k))
                pref = (D[r + k + 1][c + k + 1] - D[r][c] + D[r + 2 * k + 1][c + 1] - D[r + k][c - k]
                        + A[r + k + 1][c - k] - A[r][c + 1] + A[r + 2 * k + 1][c] - A[r + k][c + k + 1]
                        - g[r][c] - g[r + k][c + k] - g[r + 2 * k][c] - g[r + k][c - k])
                assert walk == manh == pref
                k += 1
print("mọi hình thoi: đi biên = Manhattan = tiền tố chéo")

mọi hình thoi: đi biên = Manhattan = tiền tố chéo


Edge `3×3` toàn `1`: các tổng là `1` (9 ô) và `4` (hình thoi bán kính 1) ⇒ chỉ hai giá trị phân biệt `[4, 1]`.

## 13. Lỗi người mới hay gặp

### Lỗi 1 — Đếm trùng 4 đỉnh khi cộng 4 cạnh

In [20]:
def wrong_double_corners(grid):
    m, n = len(grid), len(grid[0]); sums = set()
    for r in range(m):
        for c in range(n):
            sums.add(grid[r][c]); k = 1
            while r + 2 * k < m and c - k >= 0 and c + k < n:
                s = 0
                for t in range(k + 1):                       # SAI: mỗi cạnh k+1 ô -> đỉnh bị cộng 2 lần
                    s += grid[r + t][c + t] + grid[r + k + t][c + k - t]
                    s += grid[r + 2 * k - t][c - t] + grid[r + k - t][c - k + t]
                sums.add(s); k += 1
    return sorted(sums, reverse=True)[:3]

for g, want in examples:
    print("sai =", wrong_double_corners(g), "| đúng =", want)

sai =

 [456, 432, 422] | đúng = [228, 216, 211]
sai = [40, 9, 8] | đúng = [20, 9, 8]
sai = [7] | đúng = [7]


**Sửa:** mỗi cạnh lấy `k` ô (bỏ ô cuối) hoặc cộng `k + 1` ô rồi trừ 4 đỉnh.

### Lỗi 2 — Quên hình thoi bán kính 0 (ô đơn)

In [21]:
def wrong_no_k0(grid):
    m, n = len(grid), len(grid[0]); sums = set()
    for r in range(m):
        for c in range(n):
            k = 1                                             # SAI: bỏ qua k = 0
            while r + 2 * k < m and c - k >= 0 and c + k < n:
                sums.add(sum(grid[i][j] for i, j in border_cells_walk(r, c, k))); k += 1
    return sorted(sums, reverse=True)[:3]

for g, want in examples:
    print("sai =", wrong_no_k0(g), "| đúng =", want)

sai = [228, 216, 211] | đúng = [228, 216, 211]
sai = [20] | đúng = [20, 9, 8]
sai = [] | đúng = [7]


**Sửa:** luôn thêm `grid[r][c]`.

### Lỗi 3 — Không loại trùng (trả về 3 giá trị lớn nhất kể cả lặp)

In [22]:
def wrong_not_distinct(grid):
    vals = list(all_rhombus_sums(grid))
    return sorted(vals, reverse=True)[:3]

for g, want in examples:
    print("sai =", wrong_not_distinct(g), "| đúng =", want)

sai = [228, 216, 211] | đúng = [228, 216, 211]
sai = [20, 9, 8] | đúng = [20, 9, 8]
sai = [7, 7, 7] | đúng = [7]


**Sửa:** dùng `set` (hoặc kiểm tra `v in top`).

### Lỗi 4 — Thiếu điều kiện `c − k ≥ 0` (chỉ số âm "quấn" về cuối hàng)

Trong Python `row[-1]` là phần tử **cuối** — không báo lỗi! Nếu quên kiểm tra đỉnh trái, hình thoi "tràn" sang trái sẽ lấy nhầm ô ở mép phải và tạo ra tổng sai mà không có ngoại lệ nào.

In [23]:
def wrong_bounds(grid):
    m, n = len(grid), len(grid[0]); sums = set()
    for r in range(m):
        for c in range(n):
            sums.add(grid[r][c]); k = 1
            while r + 2 * k < m and c + k < n:                  # SAI: thiếu c - k >= 0
                sums.add(sum(grid[i][j] for i, j in border_cells_walk(r, c, k)))
                k += 1
    return sorted(sums, reverse=True)[:3]

for g, want in examples:
    print("sai =", wrong_bounds(g), "| đúng =", want)

sai = [259, 228, 216] | đúng = [228, 216, 211]
sai = [20, 19, 9] | đúng = [20, 9, 8]
sai = [7] | đúng = [7]


Ở ví dụ 1, tổng "ảo" `259` xuất hiện: một hình thoi có đỉnh trái ở cột `−1` được Python đọc thành ô ở cột cuối. Ở ví dụ 2, giá trị `19` cũng là tổng ảo. **Sửa:** kiểm tra đủ ba điều kiện `r + 2k ≤ m − 1`, `c − k ≥ 0`, `c + k ≤ n − 1`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Hình thoi trên lưới ⇔ "vòng tròn Manhattan" — biểu diễn bằng tâm + bán kính hoặc đỉnh trên + bán kính.
- Tổng trên **đường chéo** nhiều lần ⇒ **tiền tố chéo** (hai hướng `↘`, `↙`).
- Biên đa giác = các cạnh − các đỉnh (bao hàm – loại trừ); cẩn thận trường hợp suy biến (`k = 0`).
- "Top-`t` phân biệt" ⇒ `set` + sort, hoặc heap kích thước `t` + `set`.
- Lưới nhỏ (`≤ 50`) ⇒ `O(m n K²)` thường đủ — không nhất thiết phải tối ưu tới tiền tố.
- Họ bài: 1314 *Matrix Block Sum*, 1277 *Count Square Submatrices*, 498 *Diagonal Traverse*.

## 15. Biến thể của bài

1. **Tổng toàn bộ hình thoi (cả phần trong):** cộng các "vòng" bán kính `0..k` cùng tâm, hoặc dùng tiền tố trên lưới xoay 45°.
2. **Top-`t` phân biệt** với `t` tùy ý: Cách 5.
3. **Hình vuông thẳng (không xoay):** biên dùng tiền tố hàng/cột.
4. **Hình thoi có tổng lớn nhất với bán kính cố định `k`:** chỉ duyệt một `k`.
5. **Trả về tọa độ** của hình thoi đạt tổng lớn nhất.

In [24]:
def filled_rhombus_sum(grid, r, c, k):
    """Biến thể 1: tổng mọi ô bên trong + biên (tâm (r+k, c), bán kính k)."""
    cr = r + k
    return sum(grid[i][j] for i in range(len(grid)) for j in range(len(grid[0])) if abs(i - cr) + abs(j - c) <= k)

def filled_by_rings(grid, r, c, k):
    cr = r + k
    total = grid[cr][c]
    for kk in range(1, k + 1):                       # vòng bán kính kk cùng tâm, đỉnh trên (cr - kk, c)
        total += sum(grid[i][j] for i, j in border_cells_walk(cr - kk, c, kk))
    return total

print(filled_rhombus_sum(G1, 0, 2, 2), filled_by_rings(G1, 0, 2, 2))
best = max(((sum(G1[i][j] for i, j in border_cells_walk(r, c, k)), (r, c, k))
            for r in range(5) for c in range(5) for k in range(3) if fits(5, 5, r, c, k)))
print("biến thể 5 — hình thoi tổng lớn nhất:", best)

330 330
biến thể 5 — hình thoi tổng lớn nhất: (228, (1, 1, 1))


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Manhattan | quét cả lưới cho mỗi hình thoi | `O(m²n²K)` | `O(mnK)` | kiểm chứng |
| 2. Đi biên | `4k` ô mỗi hình thoi | `O(mnK²)` | `O(mnK)` | dễ viết, đủ nhanh |
| 3. Tiền tố chéo | 4 cạnh − 4 đỉnh | `O(mnK)` | `O(mn)` | **nộp bài** |
| 4. + 3 biến | top-3 tại chỗ | `O(mnK)` | `O(mn)` | tiết kiệm bộ nhớ |
| 5. Heap top-`t` | tổng quát | `O(mnK log t)` | `O(mn + t)` | khi `t` thay đổi |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `getBiggestThree([[1,2,3],[4,5,6]])` và `getBiggestThree([[2,2],[2,2],[2,2]])`?

**Bài 2 (tính tay):** với ví dụ 1, tính tổng hình thoi đỉnh trên `(0, 2)`, bán kính `2` bằng 4 cạnh − 4 đỉnh.

**Bài 3 (điền TODO):** hoàn thiện `count_rhombuses(m, n)` trả về số hình thoi hợp lệ trong lưới `m × n` (công thức hoặc đếm).

**Bài 4 (tự viết lại):** viết `biggest_t(grid, t)` trả về `t` tổng hình thoi phân biệt lớn nhất, dùng Cách 3 + heap.

**Bài 5 (mini challenge):** biến thể **hình thoi đặc** (tổng mọi ô bên trong và trên biên). Viết lời giải nhanh hơn vét cạn (gợi ý: cộng dồn các "vòng" dùng tiền tố chéo, hoặc xoay lưới 45° rồi dùng tiền tố 2D), và kiểm chứng.

In [25]:
def count_rhombuses(m, n):
    # TODO: với mỗi (r, c), đếm số k >= 0 sao cho hình thoi nằm trong lưới
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) Lưới 2×3 chỉ có hình thoi bán kính 0 (cần 3 hàng cho `k = 1`). (2) Các ô `1..6`. (3) `[6, 5, 4]`; lưới 3×2: cần `c − 1 ≥ 0` và `c + 1 ≤ 1` — không có ⇒ chỉ giá trị `2` ⇒ `[2]`.

**Bài 2:** (1) Đỉnh: `T=(0,2)=5`, `R=(2,4)=10`, `B=(4,2)=2`, `L=(2,0)=20`. (2) Cạnh `T→R`: `5,2,10`; `R→B`: `10,4,2`; `B→L` (`L→B`): `20,5,2`; `T→L`: `5,3,20`. (3) Tổng cạnh `17 + 16 + 27 + 28 = 88`, trừ đỉnh `37` ⇒ `51`.

**Bài 3:** (1) Với `(r, c)`: `k ≤ (m − 1 − r)/2`, `k ≤ c`, `k ≤ n − 1 − c`. (2) Số `k` hợp lệ = `min(...) + 1`. (3) Cộng lại.

**Bài 4:** (1) Dùng `all_rhombus_sums`. (2) `top_t_distinct(..., t)`. (3) So với `sorted(set(...))[:t]`.

**Bài 5:** (1) Hình thoi đặc bán kính `k` = hình thoi đặc bán kính `k − 1` (cùng tâm) + vòng bán kính `k`. (2) Mỗi vòng `O(1)` bằng tiền tố chéo ⇒ mỗi tâm `O(K)` cho mọi bán kính. (3) Kiểm chứng với `filled_rhombus_sum`.

## 19. Đáp án tham khảo

In [26]:
# Bài 1
assert Solution().getBiggestThree([[1, 2, 3], [4, 5, 6]]) == [6, 5, 4]
assert Solution().getBiggestThree([[2, 2], [2, 2], [2, 2]]) == [2]

# Bài 2
assert sum(G1[i][j] for i, j in border_cells_walk(0, 2, 2)) == 51

# Bài 3
def count_rhombuses_ans(m, n):
    return sum(min((m - 1 - r) // 2, c, n - 1 - c) + 1 for r in range(m) for c in range(n))
for m in range(1, 9):
    for n in range(1, 9):
        assert count_rhombuses_ans(m, n) == sum(1 for r in range(m) for c in range(n) for k in range(10) if fits(m, n, r, c, k))

# Bài 4
def biggest_t(grid, t):
    return top_t_distinct(all_rhombus_sums(grid), t)
random.seed(121)
for _ in range(200):
    m, n = random.randint(1, 8), random.randint(1, 8)
    g = [[random.randint(1, 20) for _ in range(n)] for _ in range(m)]
    t = random.randint(1, 6)
    assert biggest_t(g, t) == sorted(set(all_rhombus_sums(g)), reverse=True)[:t]
print("Bài 1-4: OK; số hình thoi trong 50x50 =", count_rhombuses_ans(50, 50))

Bài 1-4: OK; số hình thoi trong 50x50 = 22100


In [27]:
# Bài 5 — tổng hình thoi đặc cho mọi (tâm, bán kính) bằng cộng dồn vòng + tiền tố chéo
def ring_sum(D, A, g, r, c, k):
    """Tổng biên hình thoi đỉnh trên (r, c) bán kính k (k >= 0)."""
    if k == 0:
        return g[r][c]
    return (D[r + k + 1][c + k + 1] - D[r][c] + D[r + 2 * k + 1][c + 1] - D[r + k][c - k]
            + A[r + k + 1][c - k] - A[r][c + 1] + A[r + 2 * k + 1][c] - A[r + k][c + k + 1]
            - g[r][c] - g[r + k][c + k] - g[r + 2 * k][c] - g[r + k][c - k])

def all_filled_sums(g):
    m, n = len(g), len(g[0]); D, A = diag_prefix(g); out = {}
    for cr in range(m):
        for cc in range(n):
            total = 0; k = 0
            while cr - k >= 0 and cr + k < m and cc - k >= 0 and cc + k < n:
                total += ring_sum(D, A, g, cr - k, cc, k)      # vòng bán kính k, đỉnh trên (cr-k, cc)
                out[(cr, cc, k)] = total; k += 1
    return out

random.seed(122)
for _ in range(60):
    m, n = random.randint(1, 8), random.randint(1, 8)
    g = [[random.randint(1, 30) for _ in range(n)] for _ in range(m)]
    for (cr, cc, k), v in all_filled_sums(g).items():
        assert v == filled_rhombus_sum(g, cr - k, cc, k)
print("Bài 5: tổng hình thoi đặc (cộng dồn vòng) khớp vét cạn; ví dụ 1 tâm (2,2) bán kính 2 =", all_filled_sums(G1)[(2, 2, 2)])

Bài 5: tổng hình thoi đặc (cộng dồn vòng) khớp vét cạn; ví dụ 1 tâm (2,2) bán kính 2 = 330


## 20. Template nhớ nhanh + Checklist

```python
# Tiền tố hai đường chéo
D[i+1][j+1] = D[i][j]   + g[i][j]     # ↘
A[i+1][j]   = A[i][j+1] + g[i][j]     # ↙
# Hình thoi đỉnh trên (r, c), bán kính k >= 1 (cần r+2k < m, c-k >= 0, c+k < n)
s = (D[r+k+1][c+k+1] - D[r][c]          # T -> R
   + D[r+2k+1][c+1]  - D[r+k][c-k]      # L -> B
   + A[r+k+1][c-k]   - A[r][c+1]        # T -> L
   + A[r+2k+1][c]    - A[r+k][c+k+1]    # R -> B
   - (4 đỉnh))
# k = 0: g[r][c];  kết quả: sorted(set(sums), reverse=True)[:3]
```

**Checklist trước khi submit:**
- [ ] Có hình thoi bán kính 0 (mọi ô).
- [ ] Không đếm trùng 4 đỉnh.
- [ ] Điều kiện biên: `r + 2k < m`, `c − k ≥ 0`, `c + k < n` (cảnh giác chỉ số âm!).
- [ ] Giá trị **phân biệt**; trả về ít hơn 3 nếu không đủ.
- [ ] Thứ tự giảm dần.